In [19]:
import datetime
import logging
import sys

import git
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import seaborn.objects as so
from darts import TimeSeries
from darts.models import Prophet
from matplotlib import style
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer

# NOTE: the usual way of logging.getLogger(...).setLevel(...) doesn't seem to work with prophet when doing historical forecast
logging.disable(sys.maxsize)

repo = git.Repo(".", search_parent_directories=True).working_tree_dir

In [20]:
print(style.available)
PLOT_WIDTH = 16
PLOT_HEIGHT = 8
sns.set_theme(
    context="notebook",  # Options: 'paper', 'notebook', 'talk', 'poster'
    style="whitegrid",  # Options: 'darkgrid', 'whitegrid', 'dark', 'white', 'ticks'
)

['Solarize_Light2', '_classic_test_patch', '_mpl-gallery', '_mpl-gallery-nogrid', 'bmh', 'classic', 'dark_background', 'fast', 'fivethirtyeight', 'ggplot', 'grayscale', 'seaborn-v0_8', 'seaborn-v0_8-bright', 'seaborn-v0_8-colorblind', 'seaborn-v0_8-dark', 'seaborn-v0_8-dark-palette', 'seaborn-v0_8-darkgrid', 'seaborn-v0_8-deep', 'seaborn-v0_8-muted', 'seaborn-v0_8-notebook', 'seaborn-v0_8-paper', 'seaborn-v0_8-pastel', 'seaborn-v0_8-poster', 'seaborn-v0_8-talk', 'seaborn-v0_8-ticks', 'seaborn-v0_8-white', 'seaborn-v0_8-whitegrid', 'tableau-colorblind10']


## Helper Functions

In [21]:
def get_forecasted_topic_df(topic_df, method="lm", only_post=True):
    questions = topic_df["Question"].unique()
    combined_df = []
    for question in questions:
        question_df = topic_df[topic_df["Question"] == question].copy()

        pre_covid_df = question_df[question_df["Year"].dt.year < 2020].copy()
        post_covid_df = question_df[question_df["Year"].dt.year >= 2020].copy()

        question_df["Type"] = "Actual"

        X_pre = pre_covid_df["Year"].dt.year.values.reshape(-1, 1)
        if only_post:
            X_post = post_covid_df["Year"].dt.year.values.reshape(-1, 1)
        else:
            X_post = question_df["Year"].dt.year.values.reshape(-1, 1)

        if len(pre_covid_df) >= 4:
            if method == "lm":
                model = LinearRegression()
                model.fit(X_pre, pre_covid_df["DataValue"])
                pred = model.predict(X_post)
            elif method == "poly":
                poly = PolynomialFeatures(degree=2, include_bias=False)
                model = make_pipeline(poly, LinearRegression())
                model.fit(X_pre, pre_covid_df["DataValue"])
                pred = model.predict(X_post)
            elif method == "spline":
                n_points = len(pre_covid_df)
                spline = SplineTransformer(
                    n_knots=max(2, (n_points - 1) // 2),
                    degree=min(3, (n_points - 1) // 2),
                    include_bias=False,
                    extrapolation="constant",
                )
                model = make_pipeline(spline, LinearRegression())
                model.fit(X_pre, pre_covid_df["DataValue"])
                pred = model.predict(X_post)
            elif method == "prophet":
                time_series = TimeSeries.from_dataframe(
                    pre_covid_df, "Year", "DataValue"
                )
                model = Prophet()
                model.fit(time_series)
                pred = model.predict(len(post_covid_df)).values()
                if not only_post:
                    historical_pred = model.historical_forecasts(time_series).values()
                    # minimum number of training data
                    padding = np.array([np.nan] * 3).reshape(-1, 1)
                    historical_pred = np.concatenate(
                        [padding, historical_pred], axis=None
                    )
                    pred = np.concatenate((historical_pred, pred), axis=None)
            else:
                raise ValueError(f"Unknown method: {method}")

            if only_post:
                pred_df = post_covid_df.copy()
            else:
                pred_df = question_df.copy()
            pred_df["DataValue"] = pred
            pred_df["Type"] = "Predicted"

            combined_df.append(pd.concat([question_df, pred_df], ignore_index=True))
        else:
            combined_df.append(question_df)

    # combined_df is a list of dataframes
    return pd.concat(combined_df, ignore_index=True)

In [22]:
def plot_forecast_comparison(df, method="lm", only_post=True):
    df = df.copy()
    df["Year"] = pd.to_datetime(df["Year"])

    # ASSUMPTION: there's only one data_value_type per df
    data_value_type = df["DataValueType"][0]

    topics = df["Topic"].unique()
    for topic in topics:
        topic_df = df[df["Topic"] == topic]
        plot_height = PLOT_HEIGHT * len(topic_df["Question"].unique())

        forecasted_topic_df = get_forecasted_topic_df(topic_df, method, only_post)

        actual_df = forecasted_topic_df[forecasted_topic_df["Type"] == "Actual"].copy()
        predicted_df = forecasted_topic_df[
            forecasted_topic_df["Type"] == "Predicted"
        ].copy()

        plot = (
            so.Plot(forecasted_topic_df, x="Year", y="DataValue", color="Question")
            .facet(col="Question", wrap=1)
            .share(x=False, y=False)
            # separate styles are possible for actual and predicted data
            .add(
                so.Line(),
                legend=False,
                data=actual_df,
            )
            .add(
                so.Line(linestyle=(6, 2)),
                legend=False,
                data=predicted_df,
            )
            .scale(
                x=so.Temporal().tick(upto=20)
            )  # HACK: set 1 tick per year, 20 was a big enough random number
            .label(x=None, y=f"{data_value_type}")
            .layout(size=(PLOT_WIDTH, plot_height))
            .theme(sns.axes_style("whitegrid") | sns.plotting_context("notebook"))
            .plot(True)
        )

        # HACK: seaborn.objects doesn't support Rule() yet, so working with underlying figure
        # Also, because I disabled legends, have to manually add back actual and predicted
        for ax in plot._figure.axes:
            # the pound sign is the "alternate form"; causes result to always contain a decimal point and to not remove trailing zeroes https://stackoverflow.com/questions/41840613/understanding-the-python-g-in-string-formatting-achieving-java-string-format-b
            ax.yaxis.set_major_formatter(lambda y, _: f"{y:#.3g}%")

            # covid-19 line
            ax.axvline(
                x=pd.Timestamp("2020-01-01"),
                color="red",
                linestyle="-",
                alpha=1,
                label="COVID-19",
            )

            # show point of biggest difference in 2022 or 2023
            question = ax.get_title()
            years_with_both = set(
                actual_df[
                    (actual_df["Question"] == question)
                    & (actual_df["Year"].dt.year >= 2022)
                ]["Year"]
            ) & set(
                predicted_df[
                    (predicted_df["Question"] == question)
                    & (predicted_df["Year"].dt.year >= 2022)
                ]["Year"]
            )

            if years_with_both:
                diffs = []
                for year in years_with_both:
                    actual_val = actual_df[
                        (actual_df["Question"] == question)
                        & (actual_df["Year"] == year)
                    ]["DataValue"].iloc[0]
                    pred_val = predicted_df[
                        (predicted_df["Question"] == question)
                        & (predicted_df["Year"] == year)
                    ]["DataValue"].iloc[0]
                    diffs.append((year, actual_val - pred_val, actual_val))

                max_diff_year, max_diff, y_pos = max(diffs, key=lambda x: abs(x[1]))

                ax.annotate(
                    f"{max_diff:+.1f}%",
                    xy=(max_diff_year, y_pos),
                    xytext=(12, 12),
                    fontsize=11,
                    textcoords="offset points",
                    ha="left",
                    va="bottom",
                    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray"),
                    arrowprops=dict(arrowstyle="->", color="gray", linewidth=1.5),
                )

            # add custom legend box
            color = ax.get_lines()[0].get_color()
            actual_line = plt.Line2D([], [], color=color, linestyle="-", label="Actual")
            predicted_line = plt.Line2D(
                [], [], color=color, linestyle="--", label=f"Predicted ({method})"
            )
            covid_line = plt.Line2D(
                [], [], color="red", linestyle="-", label="COVID-19"
            )
            ax.legend(
                handles=[actual_line, predicted_line, covid_line],
                loc="upper left",
                frameon=True,
                edgecolor="black",
            )

            # center question in each facet
            ax.set_title(ax.get_title(), pad=10, ha="left", fontweight="bold", x=0)

        # HACK: an overarching title per plot, again need to work with underlying matplotlib figure
        plot._figure.suptitle(topic, y=1, fontweight="bold", fontsize=16)

        plot.show()

In [23]:
def plot_forecast_comparison_plotly(df, method="lm", only_post=True, interactive=True):
    df = df.copy()
    df["Year"] = pd.to_datetime(df["Year"])
    figures = []
    data_value_type = df["DataValueType"][0]
    topics = df["Topic"].unique()
    colors = px.colors.qualitative.Set2
    for topic in topics:
        topic_df = df[df["Topic"] == topic]
        forecasted_df = get_forecasted_topic_df(topic_df, method, only_post)

        # 1 topic == 1 plot
        fig = go.Figure()

        # each question == 1 line
        # legend for actual and predicted is grouped by question
        unique_questions = forecasted_df["Question"].unique()
        for i, question in enumerate(unique_questions):
            # subset data per question, and sort by year so the lines are drawn properly
            question_data = forecasted_df[
                forecasted_df["Question"] == question
            ].sort_values("Year")

            # use the same color for both actual and predicted
            line_color = colors[i % len(colors)]

            # actual line
            actual_data = question_data[question_data["Type"] == "Actual"]
            fig.add_trace(
                go.Scatter(
                    x=actual_data["Year"],
                    y=actual_data["DataValue"],
                    name=question,
                    legendgroup=question,
                    line=dict(color=line_color),
                    mode="lines",  # remove dots from lines
                )
            )

            # predicted line
            predicted_data = question_data[question_data["Type"] == "Predicted"]
            fig.add_trace(
                go.Scatter(
                    x=predicted_data["Year"],
                    y=predicted_data["DataValue"],
                    name="Predicted",
                    legendgroup=question,
                    showlegend=False,
                    line=dict(color=line_color, dash="dash"),
                    mode="lines",
                )
            )

        # formatting
        fig.update_layout(
            title=topic,
            template="plotly_white",
            height=500,
            legend=dict(
                orientation="h",  # horizontal legend
                yanchor="bottom",
                y=1.02,  # put above the plot
                xanchor="center",  # so the legend lines up side by side
                x=0.5,  # center
                font=dict(size=10),
                itemwidth=30,
                itemsizing="constant",
            ),
        )

        # y axes
        y_values = forecasted_df["DataValue"]
        y_min = y_values.min()
        y_max = y_values.max()
        y_tick_min = y_min - y_min * 0.15
        y_tick_max = y_max + y_max * 0.15

        # adjust ticks dynamically
        y_range = y_tick_max - y_tick_min
        tick_ranges = {
            range_val: tick
            for range_val, tick in [(5, 0.5), (10, 1), (20, 2), (50, 5)]
            if y_range <= range_val
        }
        y_dtick = tick_ranges[min(tick_ranges)] if tick_ranges else 10

        fig.update_yaxes(
            ticksuffix="%",
            tickmode="linear",
            tick0=0,
            dtick=y_dtick,
            title=data_value_type,
            range=[y_tick_min, y_tick_max],
        )

        # x axes
        x_values = forecasted_df["Year"]
        x_min = x_values.min()
        x_max = x_values.max()
        fig.update_xaxes(
            range=[x_min, x_max],
            dtick="M12",  # yearly interval, not sure why but Y1 breaks
            tickformat="%Y",  # only show the year, so no "Jan 2011"
        )

        # add COVID-19 vertical line
        fig.add_vline(
            # HACK: https://github.com/plotly/plotly.py/issues/3065 otherwise it throws an error if we dont convert to timestamp first
            x=datetime.datetime.strptime("2020-01-01", "%Y-%m-%d").timestamp() * 1000,
            line_color="red",
            line_dash="solid",
            annotation_text="COVID-19",
            annotation_position="top right",
        )

        figures.append(fig)

    for fig in figures:
        if interactive:
            fig.show()
        else:
            fig.show(config={"staticPlot": True, "displayModeBar": False})

In [24]:
def plot_year_over_year_growth(dataframe):
    df = dataframe.copy()
    df = df.groupby(["Question", "Year"])["DataValue"].mean().reset_index()

    growth_rates = {}
    for question in df["Question"].unique():
        question_data = df[df["Question"] == question].sort_values("Year")

        pre_covid_df = question_data[question_data["Year"] < "2020"]
        post_covid_df = question_data[question_data["Year"] >= "2020"]

        if not pre_covid_df.empty and not post_covid_df.empty:
            pre_covid_first = pre_covid_df.iloc[0]["DataValue"]
            pre_covid_last = pre_covid_df.iloc[-1]["DataValue"]
            pre_covid_growth = (
                (pre_covid_last - pre_covid_first) / pre_covid_first
            ) * 100

            post_covid_first = post_covid_df.iloc[0]["DataValue"]
            post_covid_last = post_covid_df.iloc[-1]["DataValue"]
            post_covid_growth = (
                (post_covid_last - post_covid_first) / post_covid_first
            ) * 100

            growth_rates[question] = (pre_covid_growth, post_covid_growth)

    plt.figure(figsize=(PLOT_WIDTH, PLOT_HEIGHT))

    sns.lineplot(
        data=df,
        x="Year",
        y="DataValue",
        hue="Question",
        style="Question",
        markers=True,
    )

    # sort labels by post-COVID growth
    handles, labels = plt.gca().get_legend_handles_labels()
    new_labels = [
        f"{label} (Pre-COVID: {growth_rates[label][0]:.1f}%, Post-COVID: {growth_rates[label][1]:.1f}%, Change: {growth_rates[label][1]-growth_rates[label][0]:.1f}%)"
        for label in labels
    ]

    sorted_pairs = sorted(
        zip(handles, new_labels, labels),
        key=lambda x: growth_rates[x[2]][1]
        - growth_rates[x[2]][0],  # Sort by change instead of post-COVID value
        reverse=True,
    )
    handles, new_labels, _ = zip(*sorted_pairs)

    plt.title("Year-Over-Year Growth Trends", fontsize=16)
    plt.xlabel(None)
    plt.ylabel("Growth", fontsize=14)

    ax = plt.gca()
    ax.yaxis.set_major_formatter(lambda y, _: f"{y:#.3g}%")

    ax.axvline(
        x="2020",
        color="red",
        linestyle="-",
        alpha=1,
        label="COVID-19",
    )

    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)

    plt.legend(
        handles=handles,
        labels=new_labels,
        title="Growth Rate Change",
        fontsize=8,
        title_fontsize=10,
        loc="upper left",
        frameon=True,
        edgecolor="black",
    )

    plt.grid(alpha=0.5)
    plt.tight_layout()
    plt.show()

## Prepare the dataframes

### NOTE: Here we will only need 5 columns: Year, Topic, Question, DataValue, DataValueType. There must only be one row per unique combination Topic-Question -- i.e., make sure data is only for one LocationAbbr and DataValueType

TODO: Make it more flexible

In [25]:
disability_processed_path = f"{repo}/datasets/processed/disability.csv"

disability_dtype = {
    "Year": "string",
    "Topic": "string",
    "Question": "string",
    "DataValueType": "string",
    "DataValue": "float32",
}

disability_df = pd.read_csv(disability_processed_path, dtype=disability_dtype)

disability_df.head()

,Year,Topic,Question,DataValueType,DataValue
0,2016,Disability Estimates,Self-care Disability Prevalence,Age-adjusted Prevalence,3.500000
1,2016,Disability Estimates,No Disability Prevalence,Age-adjusted Prevalence,75.400002
2,2016,Disability Estimates,Vision Disability Prevalence,Age-adjusted Prevalence,4.500000
3,2016,Disability Estimates,Mobility Disability Prevalence,Age-adjusted Prevalence,12.600000
4,2016,Disability Estimates,Any Disability Prevalence,Age-adjusted Prevalence,24.600000


In [46]:
brfss_processed_path = f"{repo}/datasets/processed/brfss.csv"

brfss_dtype = {
    "Year": "string",
    "Topic": "string",
    "Question": "string",
    "DataValueType": "string",
    "DataValue": "float32",
}

brfss_df = pd.read_csv(brfss_processed_path, dtype=brfss_dtype)

brfss_df.head()

,Year,LocationAbbr,Topic,Question,Response,Break_Out,DataValue,DataValueType
0,2011,UW,Respiratory Diseases,Current Asthma,Yes,Overall,9.1,Crude Prevalence
1,2012,UW,Respiratory Diseases,Current Asthma,Yes,Overall,8.9,Crude Prevalence
2,2013,UW,Respiratory Diseases,Current Asthma,Yes,Overall,9.0,Crude Prevalence
3,2014,UW,Respiratory Diseases,Current Asthma,Yes,Overall,8.9,Crude Prevalence
4,2015,UW,Respiratory Diseases,Current Asthma,Yes,Overall,9.2,Crude Prevalence


## Plots

In [ ]:
disability_breakdown_df = disability_df[
    (disability_df["Question"] != "No Disability Prevalence")
    & (disability_df["Question"] != "Any Disability Prevalence")
]

disability_without_no_df = disability_df[
    disability_df["Question"] != "No Disability Prevalence"
]

In [ ]:
plot_forecast_comparison(brfss_df, method="prophet", only_post=False)

In [ ]:
plot_year_over_year_growth(brfss_df)

In [ ]:
plot_forecast_comparison(disability_without_no_df, method="spline", only_post=False)

In [ ]:
plot_year_over_year_growth(disability_breakdown_df)

In [47]:
plot_forecast_comparison_plotly(brfss_df, method="spline", only_post=False)